In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
import scipy.stats as stats
from matplotlib.pylab import norm


NB_ID = "13"

In [ ]:
def load_barrage_data():
    
    print(f"Loading {MIT_BARRAGE_X.name}...")
    X = np.load(MIT_BARRAGE_X)
    Y = np.load(MIT_BARRAGE_Y)
    df = pd.read_csv(MIT_BARRAGE_DATASET_METADATA_FILE)
    
    return X, Y, df

X_barr, Y_barr, df_meta = load_barrage_data()

print(f"--- Loaded ---")
print(f"Mixtures: {X_barr.shape}")
print(f"Targets:  {Y_barr.shape}")
print(f"Metadata: {len(df_meta)} rows")

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df_meta, x='sinr_db')
ax.set_title("Distribution of Samples by SINR Level")
ax.set_xlabel("SINR (dB)")
ax.set_ylabel("Count")

# Add count labels
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 10), textcoords='offset points')

save_plot("sinr_distribution", machine_id="B",nb_id=NB_ID, fig_id="01")
plt.show()

# Verification
min_count = df_meta['sinr_db'].value_counts().min()
max_count = df_meta['sinr_db'].value_counts().max()
print(f"Balance Check: Min class count {min_count}, Max class count {max_count}")

In [ ]:
Noise_est = X_barr - Y_barr

# Calculate Power
P_clean = np.mean(np.abs(Y_barr)**2, axis=1)
P_noise = np.mean(np.abs(Noise_est)**2, axis=1)

# Measured SINR
sinr_measured = 10 * np.log10(P_clean / (P_noise + 1e-12))

# Plot
plt.figure(figsize=(6, 6))
sns.scatterplot(x=df_meta['sinr_db'], y=sinr_measured, alpha=0.3)

min_val = min(df_meta['sinr_db'].min(), sinr_measured.min())
max_val = max(df_meta['sinr_db'].max(), sinr_measured.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')

plt.title("Target vs Measured SINR (Barrage)")
plt.xlabel("Target SINR (dB)")
plt.ylabel("Measured SINR (dB)")
plt.legend()
plt.grid(True)
save_plot("barrage_sinr_physics_check", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

# Error
error = np.mean(np.abs(df_meta['sinr_db'] - sinr_measured))
print(f"Mean SINR Error: {error:.4f} dB")
assert error < 0.1, "CRITICAL: Barrage mixing math is wrong!"

# Fig 13.2 Physics Check: The "Mixing Engine"

This scatter plot validates the correction of the "Unit Power Assumption" bug found in previous sessions.

### The Data
* **X-Axis:** `sinr_target` (The requested noise level).
* **Y-Axis:** `sinr_measured` (The actual calculated noise level).
* **Red Line:** The ideal $y=x$ identity line.

### The Verdict: "Perfect Alignment" 
* **Observation:** The blue dots align tightly with the red reference line.
* **Physics Explanation:**
    * **The Fix:** By dynamically calculating the power of every slice ($P_{signal}$) before mixing, we eliminated the ~2dB drift observed in the MIT dataset.
    * **Precision:** The system now delivers the exact "Harshness" requested. A label of -10dB is physically accurate, ensuring the Autoencoder learns valid denoising gradients.

In [ ]:
def calc_sfm(signals):
    sfm_list = []
    for sig in signals:
        # Welch's PSD
        f, psd = signal.welch(sig, fs=1.0, nperseg=1024)
        psd = psd[1:] # Remove DC
        
        # SFM = Geometric Mean / Arithmetic Mean
        sfm = gmean(psd) / (np.mean(psd) + 1e-12)
        sfm_list.append(sfm)
    return np.array(sfm_list)

print("Calculating Spectral Flatness (Subset)...")
subset_n = 500
sfm_vals = calc_sfm(X_barr[:subset_n])
meta_sub = df_meta.iloc[:subset_n]

plt.figure(figsize=(10, 6))
sns.boxplot(data=meta_sub, x='sinr_db', y=sfm_vals)
plt.title("Barrage Jamming: Spectral Flatness vs SINR")
plt.xlabel("SINR (dB)")
plt.ylabel("Spectral Flatness (1.0 = Flat Noise)")
plt.grid(True, alpha=0.3)

save_plot("barrage_flatness_trend", machine_id="B",nb_id=NB_ID,fig_id="03")
plt.show()